# Ориентация текстового кропа: 0° или 180°

Модель получает кроп, найденный детектором текста, и выдаёт `p_180` — вероятность того, что
кроп перевёрнут. Метрика — `1 − Brier Score`. Обучающих данных организаторы не дают, только
20 000 тестовых картинок без меток.

**Результат: 1 − Brier = 0.95935** на тестовой выборке.

---

## Идея решения

### Метка берётся бесплатно

Размеченных данных нет, но они и не нужны. Если взять заведомо **ровную** строку текста, то
она сама — класс 0, а её же поворот на 180° — класс 1. Вся задача сводится к тому, чтобы
научиться производить ровные кропы, статистически похожие на тестовые.

Источников ровных кропов три:

| источник | откуда известна ровность | кропов |
|---|---|---|
| синтетика | рисуем текст сами, значит знаем ориентацию | 550 016 |
| HierText | разметка задаёт четырёхугольник, нулевая вершина — левый верхний угол *текста* | 208 108 |
| TextOCR | то же соглашение о вершинах, слова склеены в строки | 781 655 |

В HierText и TextOCR угол верхнего ребра четырёхугольника сам говорит, ровная строка в кадре
или перевёрнутая. Распознаватель для получения метки не нужен вообще. Соглашение проверено по
самой разметке: у 94.6% четырёхугольников первое ребро направлено вправо, у 94.4% второе —
вниз; остаток — действительно наклонённый и вертикальный текст, который отбрасывается.

### Граница поворота — главный инвариант пайплайна

Аугментации делятся на две группы, и порядок между ними нарушать нельзя:

    строка текста -> шрифт -> цвет -> рендер -> подложка -> компоновка
    -> перспектива, размытие, яркость, наклон, дуга
    ------------------------ ГРАНИЦА ПОВОРОТА ------------------------
    -> поворот на 180° согласно метке
    -> джиттер разрешения, шум, JPEG

Всё, что симметрично относительно поворота, применяется **до** него. Всё несимметричное —
прежде всего сетка блоков JPEG, которая всегда отсчитывается от левого верхнего угла, —
строго **после**. Иначе класс угадывался бы по артефактам сжатия, а не по тексту, и модель
показывала бы прекрасные метрики на валидации и случайные на тесте.

### Подгонка под тестовую выборку

Пиксели теста в обучении не используются. Используются только **распределения ширины и высоты**
его кропов. Это не утечка: поворот на 180° не меняет ни ширину, ни высоту, поэтому извлечь из
геометрии метку нельзя даже в принципе.

По этому профилю выравниваются обе величины сразу — высота кропа и его пропорции. Выравнивать
только высоту оказалось мало и опасно: пока метрика взвешивалась по одной высоте, диапазону
`aspect < 2.5` доставалось 30% веса при 7% в тесте, а это самый трудный для модели диапазон.
Валидация занижала себя — показывала 0.9242 при 0.9473 на тесте. После совместного выравнивания
та же модель даёт 0.9430, и остаток разрыва 0.004.

Обучающая выборка дополнительно ограничивает число повторов одного кропа (усечённое
importance-семплирование, Ionides 2008). Без потолка вся масса садится на горстку кропов из
редких ячеек: 782 тысячи кропов TextOCR вырождались в эффективные 7 тысяч. Потолок в 10
повторов поднимает эффективный размер до 100 тысяч, а расхождение с тестовым распределением
пропорций растёт всего с 0.04 до 0.08 по полной вариации.

### Симметризация предсказания

Для любого кропа ровно одна из двух ориентаций верна, поэтому обязано выполняться
`p(x) + p(rot180 x) = 1`. Обученная сеть это соотношение лишь приближает. Мы навязываем его
тождественно, усредняя логиты прямого и перевёрнутого прогона:

    logit(x) = (z(x) − z(rot180 x)) / 2

Результат антисимметричен по построению. Стоит это двойного прогона сети, но сеть — 22 тысячи
параметров, её прогон занимает 2 секунды на все 20 000 кропов.

### Калибровка

Brier наказывает переуверенность квадратично, поэтому логиты делятся на температуру, подобранную
**прямым минимумом Brier** (а не log-loss) на отложенной половине валидации. Калибровать на тех
же кропах, по которым потом считается балл, нельзя: замер показал завышение около 0.015 —
больше, чем разница между вариантами модели.

---

## Архитектура

Своя сеть на разделимых свёртках, два варианта в ансамбле:

| модель | параметров | MAC на кроп | высота входа | вертикальное прореживание |
|---|---|---|---|---|
| `tall_v2` | 23 777 | 10.3M | 48 | 4 |
| `tiny_v2` | 22 241 | 8.4M | 48 | 8 |

Две особенности, обе продиктованы природой сигнала.

**Высота сворачивается в каналы, а не усредняется.** Различающий признак — вертикальная
асимметрия: выносные элементы, положение базовой линии, высота точек и запятых. Глобальное
усреднение по картинке уничтожило бы информацию «на какой высоте находится признак» — ровно ту,
которая отличает `р` от `ь`.

**Усреднение идёт только по ширине, среднее вместе с максимумом.** Ориентация — свойство всей
строки, и каждая буква даёт независимое свидетельство; усреднение по ширине их складывает.
Максимум добавлен потому, что одного отчётливого выносного элемента достаточно, а среднее такой
одиночный признак размывает.

**Ансамбль** усредняет вероятности, а не логиты: Brier строго выпукла, поэтому по неравенству
Йенсена ошибка среднего не превышает среднюю ошибку участников — ансамбль не может быть хуже
среднего участника. Обе модели работают с одной геометрией входа, поэтому тест читается и
препроцессится один раз на обе: 19.2 секунды против 18.9 у одной модели.

---

## Как валидировались

Пять срезов, у каждого своя роль:

| срез | что показывает | кропов |
|---|---|---|
| `real_scene` | основная метрика: строки со сцен HierText, геометрия выровнена под тест | 38 991 |
| `real_hand` | рукописный текст | 3 381 |
| `cyr_plate` | кириллица с номерных знаков | 1 500 |
| `cyr_hand` | кириллическая рукопись | 1 538 |
| `synthetic` | holdout синтетики: ловит поломки генератора | 20 000 |

Валидационный сплит HierText строго отделён от обучающего, а 26 снимков TextOCR, совпадающих
с валидацией (оба датасета выросли из Open Images), исключены — иначе одна фотография попала бы
и в обучение, и в метрику.

Итоговые числа на полных срезах, калибровка на отложенной половине:

| модель | real_scene | real_hand | cyr_plate | cyr_hand |
|---|---|---|---|---|
| `tall_v2` | 0.9476 | 0.9517 | 0.9995 | 0.9357 |
| `tiny_v2` | 0.9452 | 0.9575 | 0.9990 | 0.9286 |
| **ансамбль** | **0.9518** | **0.9599** | 0.9996 | 0.9383 |

Валидация предсказала 0.9561 на тесте, получилось 0.95935.

Дополнительно — безметочная диагностика прямо на тесте: у согласованной модели
`p(x) + p(rot180 x) = 1`, и отклонение от этого равенства считается по **несимметризованному**
прогону на всех 20 000 кропов без единой метки.

---

## Что пробовали и что не сработало

**Выравнивание только по высоте** — дало прибавку, но усилило перекос по пропорциям. Исправлено
совместным выравниванием по высоте и пропорциям.

**Вход высотой 64 вместо 48.** Гипотеза: крупные кропы теряют детали при сжатии до 48, ведь
корзина «высота 80+» несёт 36% потерь при 21% веса. Результат отрицательный — крупным кропам не
помогло (−0.003), мелким навредило (−0.011). Разбор показал, что гипотеза была неверна: крупный
текст на фотографии — это вывеска или заголовок, то есть одно-два слова, и три четверти таких
кропов короче `aspect 4`. «Крупные трудные» оказалось «короткими трудными» в маскировке.

**MobileNetV3-Small с предобучением.** Отстаёт на каждом срезе, а температура калибровки растёт
с 0.74 до 1.98 за семь эпох — модель уверенно ошибается на невиденных данных. При 1.5 млн
параметров против 22 тысяч это ожидаемо, а для Brier переуверенность — худший способ ошибаться.

**Третья модель в ансамбле** (вход 64) дала на тесте +0.0009 ценой удвоения времени инференса:
у неё другая геометрия входа, поэтому тест приходится читать дважды. Отказались.

**Обрезка вероятностей** к `[eps, 1-eps]` — измерена и отвергнута: лишь 0.14% уверенных
предсказаний ошибочны, балл не меняется до пятого знака.

## Чего в решении нет

Псевдоразметки теста. Автоматическая псевдоразметка правилами не запрещена (запрещена ручная),
и на 20 000 кропов ровно из целевого распределения она могла бы дать заметную прибавку, но мы
сознательно от неё отказались.


## Как получить `submission.csv`

1. Установить пакет: `pip install -e .` в корне репозитория.
2. Положить тестовые кропы в `test/images/` — 20 000 файлов `test_XXXXX.png`.
3. Выполнить ячейки ниже: они запишут `submission.csv` в корень репозитория.

Веса обеих моделей лежат в репозитории (`artifacts/*/best.pt`, по 110 КБ), скачивать ничего не
нужно. Предсказание занимает около 20 секунд на GPU и около 40 на CPU. Обучение с нуля описано
в `README.md`.

**Про устройство.** Отправленный `submission.csv` получен на видеокарте (CUDA, RTX 2060).
Ячейки ниже сами выберут GPU, если он доступен, иначе посчитают на процессоре. Результат тот же
с точностью до шестого знака: CPU и CUDA складывают числа в разном порядке, и значения ровно на
границе округления `%.6f` расходятся на единицу последнего разряда. На нашем прогоне так
разошлись 713 строк из 20 000, ни одно предсказание не сменило сторону, сдвиг балла не
превышает `7e-8`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from avitocv.data.datasets import CenterWindowFit, CropInferenceDataset, ImagePreprocessor, PreprocessConfig
from avitocv.model.architecture import ModelCostMeter, ModelFactory, TinyNetConfig
from avitocv.model.inference import (
    DirectPredictor,
    ProbabilityEnsemble,
    TemperatureScaler,
    rotate_half_turn,
)
from avitocv.training.metrics import ConsistencyReport

TEST_IMAGES = Path("test/images")
OUTPUT = Path("submission.csv")

# Обе модели обучались на входе 48x192 в градациях серого. Значения заданы здесь явно, а не
# читаются из configs/data.yaml, чтобы предсказание не зависело от конфигурации обучения.
PREPROCESS = PreprocessConfig(height=48, width=192, channel_count=1)

# Участники ансамбля: путь к весам и вертикальное прореживание, с которым модель обучалась.
MEMBERS = [
    ("artifacts/tall_v2/best.pt", 4),
    ("artifacts/tiny_v2/best.pt", 8),
]

# Фиксированный сид: предсказание обязано быть побитово воспроизводимым.
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"устройство: {device}")
print(f"кропов найдено: {len(list(TEST_IMAGES.glob('*.png')))}")

устройство: cuda
кропов найдено: 20000


### Загрузка моделей


In [2]:
def load_member(checkpoint: str, height_downsample: int):
    """Восстанавливает модель и её температуру калибровки из чекпоинта.

    Архитектура записана в сам чекпоинт, поэтому её не надо помнить и передавать руками:
    несовпадение дало бы невнятную ошибку загрузки весов.
    """
    payload = torch.load(checkpoint, map_location=device, weights_only=True)
    config = TinyNetConfig(input_height=PREPROCESS.height, height_downsample=height_downsample)
    model = ModelFactory(config).create(payload["architecture"])
    model.load_state_dict(payload["state_dict"])
    return model.to(device).eval(), TemperatureScaler(float(payload["temperature"]))


models, scalers = [], []
shape = (PREPROCESS.channel_count, PREPROCESS.height, PREPROCESS.width)
for checkpoint, downsample in MEMBERS:
    model, scaler = load_member(checkpoint, downsample)
    cost = ModelCostMeter().measure(model.cpu(), shape)
    model.to(device)
    models.append(model)
    scalers.append(scaler)
    print(f"{checkpoint}: {cost.describe()}, температура {scaler.temperature:.3f}")

total = sum(sum(parameter.numel() for parameter in model.parameters()) for model in models)
print(f"всего параметров в ансамбле: {total}")

artifacts/tall_v2/best.pt: параметров 23 777 (0.09 МБ fp32)  MAC на кроп 10.34M, температура 0.873
artifacts/tiny_v2/best.pt: параметров 22 241 (0.08 МБ fp32)  MAC на кроп 8.45M, температура 0.812
всего параметров в ансамбле: 46018


### Прогон по тесту


In [3]:
# Тест читается ОДИН раз на обе модели: у них общая геометрия входа, а всё время предсказания
# уходит именно на чтение и препроцессинг 20 000 PNG (15 секунд против 2 на прогон сети).
dataset = CropInferenceDataset.from_directory(TEST_IMAGES, ImagePreprocessor(PREPROCESS, CenterWindowFit()))
forward = [[] for _ in models]
flipped = [[] for _ in models]

with torch.no_grad():
    for images in DataLoader(dataset, batch_size=512, num_workers=4):
        images = images.to(device)
        rotated = rotate_half_turn(images)
        for index, model in enumerate(models):
            predictor = DirectPredictor(model)
            forward[index].append(predictor.logits(images).float().cpu().numpy())
            flipped[index].append(predictor.logits(rotated).float().cpu().numpy())

forward = [np.concatenate(chunks) for chunks in forward]
flipped = [np.concatenate(chunks) for chunks in flipped]
print(f"прогон завершён: {len(forward[0])} кропов")

прогон завершён: 20000 кропов


### Симметризация, калибровка, ансамбль


In [4]:
# Симметризация: логит антисимметричен по построению, поэтому p(x) + p(rot180 x) = 1 точно.
probabilities = [
    scaler.probabilities((straight - reversed_) / 2.0)
    for scaler, straight, reversed_ in zip(scalers, forward, flipped)
]

# Усредняются вероятности, а не логиты: Brier строго выпукла, поэтому по неравенству Йенсена
# ошибка среднего не превышает среднюю ошибку участников.
ensemble = ProbabilityEnsemble().combine(probabilities)

frame = pd.DataFrame({"image_id": dataset.image_ids, "p_180": ensemble})
frame.to_csv(OUTPUT, index=False, float_format="%.6f")
print(f"записано: {OUTPUT.resolve()}")
frame.head()

записано: D:\working\AvitoCV\submission.csv


,image_id,p_180
0,test_00000,0.017878
1,test_00001,0.999999
2,test_00002,0.999980
3,test_00003,0.017567
4,test_00004,0.997956


### Проверки


In [5]:
# Безметочная диагностика. Для любого кропа ровно одна ориентация верна, поэтому у
# согласованной модели p(x) + p(rot180 x) = 1. У симметризованного предсказания это выполняется
# тождественно, поэтому проверять надо ПРЯМОЙ прогон — иначе диагностика измеряла бы ноль.
for (checkpoint, _), scaler, straight, reversed_ in zip(MEMBERS, scalers, forward, flipped):
    report = ConsistencyReport.from_pairs(scaler.probabilities(straight), scaler.probabilities(reversed_))
    print(f"{Path(checkpoint).parent.name}: {report.describe()}")

assert len(frame) == 20_000, f"ожидалось 20 000 строк, получено {len(frame)}"
assert frame["p_180"].between(0.0, 1.0).all(), "вероятности вышли за [0, 1]"
assert not frame["p_180"].isna().any(), "в предсказаниях есть пропуски"
print(f"средняя вероятность {ensemble.mean():.4f} (симметризация даёт ровно 0.5 в пределе)")
print(f"уверенных (|p-0.5| > 0.4): {np.mean(np.abs(ensemble - 0.5) > 0.4):.1%}")

tall_v2: несогласованность 0.0925 (макс 0.999), p̄ 0.498, уверенных 77.4%, n 20000
tiny_v2: несогласованность 0.0755 (макс 0.981), p̄ 0.500, уверенных 79.1%, n 20000
средняя вероятность 0.4990 (симметризация даёт ровно 0.5 в пределе)
уверенных (|p-0.5| > 0.4): 80.9%
